In [2]:
# ── Setup Cell 1: Clone repo and install package ─────────────────────────────
# Make sure you have switched the Colab runtime setting to GPU T4 
import os, subprocess, sys

REPO_URL = "https://github.com/violasarah2000/hackingrongo.git"
REPO_DIR = "/kaggle/working/repo"   # change if you cloned to a different path

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    # Fresh session — clone the repo
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    # Repo already present — fetch latest and hard-reset to match remote
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])
    print("Repo updated to latest commit.")

os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "-q"])
print("Package installed. Working directory:", os.getcwd())

Cloning into '/kaggle/working/repo'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.1/892.1 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.6 MB/s eta 0:00:00
Package installed. Working directory: /kaggle/working/repo


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [ ]:
# ── Setup Cell 2: Install extra dependencies ─────────────────────────────────
import subprocess, sys  # re-import in case this cell runs standalone
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "scipy>=1.12,<2.0",
    "scikit-image",
    "opencv-python-headless",
    "cairosvg",
    "umap-learn",
    "hdbscan",
    "matplotlib",
    "dimod>=0.12",
    "dwave-samplers>=1.0",
])
print("Extra dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.9 MB/s eta 0:00:00
Extra dependencies installed.


In [ ]:
# ── Setup Cell 3: Download and extract data ──────────────────────────────────
import subprocess, shutil, zipfile, os
from pathlib import Path

# ── Kaggle: download via gdown instead of mounting Drive ─────────────────────
DRIVE_FILE_ID = "1NGg7459yIo1Xky-rvDBwSMqZ0rYTFb_X"   # ← paste ID from your Drive share link
ZIP_PATH = Path("/kaggle/working/hackingrongo_colab.zip")  # ← changed
DATA_DIR = Path(REPO_DIR) / "data"
FORCE_REEXTRACT = False

# Download zip if not already present
if not ZIP_PATH.exists():
    print(f"Downloading zip from Google Drive...")
    subprocess.check_call([
        "gdown", DRIVE_FILE_ID, "-O", str(ZIP_PATH)
    ])
    print("Download complete.")
else:
    print("Zip already present — skipping download.")

# ── Everything below is identical to your Colab cell ─────────────────────────
_sentinel = DATA_DIR / "corpus" / "A.json"
if _sentinel.exists() and not FORCE_REEXTRACT:
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    print(f"Corpus already extracted ({n_corpus} JSON files) — skipping.")
    print(f"  Set FORCE_REEXTRACT = True to overwrite.")
else:
    TMPDIR = Path("/kaggle/working/hackingrongo_extract")  # ← changed
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR.mkdir()
    print(f"Extracting data/ from {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        data_members = [m for m in zf.namelist() if m.startswith("data/")]
        if not data_members:
            raise RuntimeError(
                "Zip contains no data/ folder. Check the zip was created from "
                "inside the hackingrongo/ project root."
            )
        zf.extractall(TMPDIR, members=data_members)
    extracted_data = TMPDIR / "data"
    DATA_DIR.mkdir(exist_ok=True)
    for item in extracted_data.iterdir():
        dest = DATA_DIR / item.name
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        shutil.move(str(item), str(dest))
    shutil.rmtree(TMPDIR)
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    svgs = list(DATA_DIR.rglob("*.svg"))
    pngs = list(DATA_DIR.rglob("*.png"))
    print(f"Done: {n_corpus} corpus JSONs, {len(svgs)} SVGs, {len(pngs)} PNGs in {DATA_DIR}")

Downloading...
From (original): https://drive.google.com/uc?id=1ywwTdqvFvrTV6i6zjBJkiWLT4GjmKDn4
From (redirected): https://drive.google.com/uc?id=1ywwTdqvFvrTV6i6zjBJkiWLT4GjmKDn4&confirm=t&uuid=a5b5ea85-9042-499a-8e80-20f7d98a99da
To: /kaggle/working/hackingrongo_colab.zip
100%|██████████| 24.8M/24.8M [00:00<00:00, 136MB/s] 


Download complete.
Extracting data/ from /kaggle/working/hackingrongo_colab.zip ...
Done: 26 corpus JSONs, 4486 SVGs, 9733 PNGs in /kaggle/working/repo/data


In [5]:
# ── Setup Cell 4: Smoke test ─────────────────────────────────────────────────
import hackingrongo

from hackingrongo.zone_a.autoencoder import ConvAutoencoder
from hackingrongo.data.dataset import GlyphImageDataset

print("hackingrongo package path:", hackingrongo.__file__)
print("Import OK — ready to train.")

hackingrongo package path: /kaggle/working/repo/hackingrongo/__init__.py
Import OK — ready to train.


In [ ]:
# ── Zone A Cell 1: Train autoencoder ─────────────────────────────────────────
# Auto-resumes from the latest checkpoint in /kaggle/working/checkpoints.
# If 50 epochs are already complete, skips training and re-extracts embeddings.
# To force a full retrain from scratch, set resume_checkpoint to "none" below
# or delete autoencoder_epoch*.pt from the checkpoints dir.
#
# Outputs:
#   /kaggle/working/checkpoints/autoencoder_epoch*.pt
#   /kaggle/working/checkpoints/embeddings_cache.pt

import os, subprocess, sys
import torch
from pathlib import Path

REPO_DIR         = "/kaggle/working/repo"
CHECKPOINTS_DIR  = "/kaggle/working/checkpoints"
EMBEDDINGS_CACHE = Path(f"{CHECKPOINTS_DIR}/embeddings_cache.pt")
DATA_DIR         = f"{REPO_DIR}/data"

Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

# ── Pre-flight: validate or purge existing embeddings cache ───────────────────
print("=" * 60)
print("PRE-FLIGHT: Checking embeddings cache")
print("=" * 60)

CACHE_VALID = False
if EMBEDDINGS_CACHE.exists():
    try:
        _d = torch.load(EMBEDDINGS_CACHE, map_location="cpu", weights_only=True)
        _has_embeddings = isinstance(_d, dict) and "embeddings" in _d
        _has_codes      = _has_embeddings and any(_d.get("barthel_codes", []))
        if _has_embeddings and _has_codes:
            _n       = len(_d["embeddings"])
            _n_coded = sum(1 for c in _d["barthel_codes"] if c)
            print(f"  Cache OK: {_n} embeddings, {_n_coded} with Barthel codes.")
            print("  Skipping training — delete embeddings_cache.pt to retrain.")
            CACHE_VALID = True
        else:
            reason = []
            if not _has_embeddings: reason.append("missing 'embeddings' key")
            if not _has_codes:      reason.append("barthel_codes empty")
            print(f"  Cache invalid ({'; '.join(reason)}) — deleting and retraining.")
            EMBEDDINGS_CACHE.unlink()
        del _d
    except Exception as e:
        print(f"  Cache unreadable ({e}) — deleting and retraining.")
        EMBEDDINGS_CACHE.unlink()
else:
    print("  No cache found — will train (auto-resumes from latest checkpoint if any).")

# ── Training ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Training Zone A autoencoder")
print("=" * 60)

if CACHE_VALID:
    print("  Skipped (valid cache already present).")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_autoencoder.py",
            f"paths.glyphs_dir={DATA_DIR}/glyphs",
            f"paths.corpus_dir={DATA_DIR}/corpus",
            f"paths.catalog_dir={DATA_DIR}/catalog",
            f"paths.checkpoints_dir={CHECKPOINTS_DIR}",
            f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
            "zone_a.autoencoder.num_epochs=50",
            "zone_a.autoencoder.batch_size=64",
            # "auto" globs checkpoints_dir for the latest .pt and resumes.
            # Change to "none" to force a full retrain from scratch.
            "zone_a.autoencoder.resume_checkpoint=auto",
            "hydra.job.chdir=false",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        shell=False,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Training failed (exit {proc.returncode})")
    print(f"\nDone. Embeddings cache at: {EMBEDDINGS_CACHE}")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

ckpts = sorted(Path(CHECKPOINTS_DIR).glob("autoencoder_epoch*.pt"))
for f in ckpts:
    print(f"  ✓ {f.name}  {f.stat().st_size/1024/1024:.1f} MB")
if not ckpts:
    print("  ✗ No checkpoints found")

if EMBEDDINGS_CACHE.exists():
    print(f"  ✓ embeddings_cache.pt  {EMBEDDINGS_CACHE.stat().st_size/1024/1024:.1f} MB")
    print("\nZone A Cell 2 is ready to run.")
else:
    print("  ✗ embeddings_cache.pt NOT FOUND — check logs above")

In [ ]:
# ── Zone A Cell 2: Analyse embeddings — UMAP + HDBSCAN + reports ─────────────
# Produces: umap_embeddings.png, cluster_vs_barthel.csv/.json,
#           divergence_report.html, compound_candidates.json,
#           compound_report.html
# Only run after Zone A Cell 1 prints "Zone A Cell 2 is ready to run."

import subprocess, sys, os, json
from pathlib import Path
from IPython.display import Image, display

REPO_DIR         = "/kaggle/working/repo"
CHECKPOINTS_DIR  = "/kaggle/working/checkpoints"
EMBEDDINGS_CACHE = f"{CHECKPOINTS_DIR}/embeddings_cache.pt"
DATA_DIR         = f"{REPO_DIR}/data"
ANALYSIS_OUT     = f"{REPO_DIR}/outputs/analysis"

os.chdir(REPO_DIR)
Path(ANALYSIS_OUT).mkdir(parents=True, exist_ok=True)

if not Path(EMBEDDINGS_CACHE).exists():
    raise FileNotFoundError("embeddings_cache.pt not found. Run Zone A Cell 1 first.")
print(f"Cache: {Path(EMBEDDINGS_CACHE).stat().st_size/1024/1024:.1f} MB  ✓")

# ── Step 1: UMAP + HDBSCAN + divergence report ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Embedding analysis + divergence report")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "scripts/analyze_embeddings.py",
        f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
        f"paths.outputs_dir={REPO_DIR}/outputs",
        "hydra.job.chdir=false",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Analysis failed (exit {proc.returncode})")

# ── Step 2: Compound detector — validation run ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Compound detector — P@k validation on known compounds")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_validation.json",
        "--include-known", "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Validation run failed (exit {proc.returncode}) — continuing")

val_path = Path(ANALYSIS_OUT) / "compound_validation.json"
if val_path.exists():
    val   = json.loads(val_path.read_text())
    cands = val.get("candidates", [])
    print("\nPrecision@k on known compounds:")
    for k in [5, 10, 20]:
        top     = cands[:k]
        n_known = sum(1 for c in top if c.get("is_known_compound", False))
        p       = n_known / max(len(top), 1)
        bar     = "█" * round(p * 20) + "░" * (20 - round(p * 20))
        print(f"  P@{k:2d}: {p:.2f}  {bar}  ({n_known}/{len(top)})")

# ── Step 3: Compound detector — novel candidates ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Compound detector — novel candidates")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_candidates.json",
        "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Detection failed (exit {proc.returncode}) — continuing")

# ── Step 4: Compound report ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Compound report")
print("=" * 60)

if (Path(ANALYSIS_OUT) / "compound_candidates.json").exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.compound_report",
            "--candidates", "outputs/analysis/compound_candidates.json",
            "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
            "--output", "outputs/analysis/compound_report.html",
            "--max-candidates", "50",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
else:
    print("No compound_candidates.json — skipping report")

# ── Step 5: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Parallel passage cross-reference")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

# ── Step 6: Diachronic passage report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: Diachronic passage report")
print("=" * 60)

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",  str(variants_path),
            "--output", f"{REPO_DIR}/outputs/analysis/passage_reports",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("parallel_variants_auto.json not found — skipping passage report")
    
# ── Step 7: Display cluster summary + UMAP ───────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Results")
print("=" * 60)

json_path = Path(ANALYSIS_OUT) / "cluster_vs_barthel.json"
if json_path.exists():
    result = json.loads(json_path.read_text())
    print(f"  Embeddings : {result['n_embeddings']}")
    print(f"  Clusters   : {result['n_clusters']}")
    print(f"  Noise pts  : {result['n_noise_points']}")
    ari = result.get("adjusted_rand_index")
    if ari is not None:
        print(f"  ARI        : {ari:.4f}  ({result.get('interpretation', '')})")
    print("\nTop 5 clusters by size:")
    clusters = {k: v for k, v in result.get("clusters", {}).items() if k != "noise"}
    for cid, info in sorted(clusters.items(), key=lambda x: -x[1]["size"])[:5]:
        top_codes = list(info["barthel_codes"].keys())[:4]
        top_fams  = list(info["barthel_families"].keys())[:2]
        print(f"  Cluster {cid}: {info['size']} glyphs  codes={top_codes}  families={top_fams}")

plot_path = Path(ANALYSIS_OUT) / "umap_embeddings.png"
if plot_path.exists():
    print("\nUMAP scatter:")
    display(Image(filename=str(plot_path)))

# ── Step 8: Astronomical glyph candidate analysis ────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Astronomical glyph candidate analysis")
print("=" * 60)

Path(f"{REPO_DIR}/outputs/zone_b").mkdir(parents=True, exist_ok=True)
proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.astronomical_analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output",     f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Astronomical analysis failed (exit {proc.returncode}) — continuing")

# Print top candidates inline
astro_path = Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json")
if astro_path.exists():
    try:
        astro_data  = json.loads(astro_path.read_text())
        candidates  = astro_data.get("candidates", [])
        print(f"\n  Candidates flagged: {len(candidates)}")
        if candidates:
            print(f"  {'Code':<8} {'Score':>6}  {'Methods':>7}  Details")
            print(f"  {'-'*8} {'-'*6}  {'-'*7}  {'-'*35}")
            for c in candidates[:10]:
                code   = c.get("barthel_code", "?")
                score  = c.get("overall_score", c.get("consensus_confidence", 0))
                n      = c.get("n_methods_flagged", c.get("n_methods_agreeing", 0))
                entry  = c.get("dietrich_entry")          # may be None
                if isinstance(entry, dict):
                    desc = entry.get("proposed_referent", "—")
                elif entry is not None:
                    desc = str(entry)
                else:
                    desc = "bird-headed / no Dietrich entry"
                print(f"  {code:<8} {score:>6.3f}  {n:>7}  {desc}")
    except Exception as e:
        print(f"  WARNING: Could not display candidates inline: {e}")
        print(f"  (astronomical_candidates.json was written — Step 9 will render it properly)")
        
# ── Step 9: Astronomical HTML report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: Astronomical report")
print("=" * 60)

if astro_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.astronomical_report",
            "--candidates", str(astro_path),
            "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
            "--output",     f"{REPO_DIR}/outputs/analysis/astronomical_report.html",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Astronomical report failed (exit {proc.returncode}) — continuing")
else:
    print("astronomical_candidates.json not found — skipping report")

# ── Step 10: Train sequence model (for glyph reconstruction) ─────────────────
print("\n" + "=" * 60)
print("STEP 10: Train sequence model")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/train_sequence_model.py",
        "--output",     f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
        "--order",      "3",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Sequence model training failed — continuing")
else:
    # Run reconstruction on Tablet F (81 glyphs, several uncertain readings)
    proc2 = subprocess.Popen(
        [
            sys.executable, "scripts/complete_sequence.py",
            "--tablet",  "F",
            "--model",   f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
            "--corpus-dir", f"{DATA_DIR}/corpus",
            "--out",     f"{REPO_DIR}/outputs/zone_b/tablet_f_reconstruction.json",
            "--top-k",   "5",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc2.stdout:
        print(line, end="")
    proc2.wait()

# ── Step 8b: Quantum p_good measurement ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8b: Quantum hardness analysis (p_good measurement)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/measure_pgood.py",
        "--corpus-dir",      f"{DATA_DIR}/corpus",
        "--lm-dir",          f"{REPO_DIR}/data/language_models",
        "--n-samples",       "10000",
        "--thresholds",      "0.90,0.95,0.99",
        "--mcmc-iterations", "5000",
        "--output",          f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: p_good measurement failed — continuing")

pgood_path = Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json")
if pgood_path.exists():
    pg = json.loads(pgood_path.read_text())
    print(f"\n  ── Quantum hardness summary ──────────────────────────────")
    print(f"  Score distribution: mean={pg['score_distribution']['mean']:.1f} "
          f"std={pg['score_distribution']['std']:.1f}")
    for t in pg['thresholds']:
        print(f"  τ={t['tau']:.2f}: p_good={t['p_good']:.2e}  "
              f"Grover={t['grover_oracle_calls']:,} calls  "
              f"Classical={t['classical_random_calls']:,}  "
              f"Speedup={t['quantum_speedup_ratio']:.1f}x")

print("\nZone C Cell 1 is ready to run.")

In [ ]:
# ── Zone B Cell 1: Build language models and parallel passages ──────────────────
# Run once after Zone A Cell 2. Takes ~5 minutes.
# Language models are required for Zone C Cell 1.
# Parallel passage cross-reference finds multi-tablet attestations.
#
# Outputs written to Drive:
#   parallel_variants_auto.json   ← 13 multi-tablet passages found
#   pre_contact_lm.json           ← Thomson 1891 + Roussel 1908 (~1345 forms)
#   post_contact_lm.json          ← Barthel/Blixen/Fischer + IDS (~2754 forms)
#   smoothing_lm.json             ← Hawaiian Corpus Project (~56K types)

import subprocess, sys, json, shutil
from pathlib import Path

REPO_DIR        = "/kaggle/working/repo"
CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

# ── Step 1: Fetch IDS Rapa Nui vocabulary ─────────────────────────────────────
print("=" * 60)
print("STEP 1: Fetch IDS Rapa Nui vocabulary (Thomson + Roussel)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/parse_ids.py",
        "--stratify",
        "--cache-dir", f"{DATA_DIR}/cache",
        "--data-dir",  f"{DATA_DIR}/polynesian_texts",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: parse_ids.py failed (exit {proc.returncode}) — continuing")

cognate_file = Path(f"{DATA_DIR}/polynesian_texts/rapanui/abvd_cognate_neighbours.txt")
if cognate_file.exists():
    print(f"  ✓ abvd_cognate_neighbours.txt: {len(cognate_file.read_text().splitlines())} forms")
else:
    print("  ✗ abvd_cognate_neighbours.txt not found — network may be blocked")
    print("    Pre-contact LM will use IDS only (~1345 forms, still functional)")

# ── Step 2: Fetch ABVD cognate neighbours ─────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Fetch ABVD East Polynesian cognate neighbours")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_abvd_corpus.py",
        "--with-cognates",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_abvd_corpus.py failed (exit {proc.returncode}) — continuing")

# ── Step 2b: Fetch Hawaiian smoothing corpus ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2b: Fetch Hawaiian smoothing corpus")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_hawaiian_corpus.py",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_hawaiian_corpus.py failed (exit {proc.returncode}) — continuing")

# ── Step 3: Build language models ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Build stratified language models")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/build_language_models.py",
        "hydra.job.chdir=false",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"build_language_models.py failed (exit {proc.returncode})")

# Gate check
print("\n── Language model verification ──────────────────────────────")
lm_dir = Path(f"{REPO_DIR}/data/language_models")
lm_ok = True
for lm_name in ["pre_contact_lm", "post_contact_lm", "smoothing_lm"]:
    p = lm_dir / f"{lm_name}.json"
    if p.exists():
        vocab = len(json.loads(p.read_text()).get("vocab", []))
        status = "✓" if vocab >= 50 else "✗ TOO SPARSE"
        print(f"  {status} {lm_name}: vocab={vocab} types")
        if vocab < 50:
            lm_ok = False
    else:
        print(f"  ✗ {lm_name}.json MISSING")
        lm_ok = False

if not lm_ok:
    print("\nWARNING: LMs too sparse — Zone C will produce uninformative results")
else:
    print("\nLMs ready for Zone C.")

# ── Step 4: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Parallel passage cross-reference")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    v = json.loads(variants_path.read_text())
    n_multi = sum(1 for p in v.get("passages", []) if p.get("n_tablets", 0) >= 2)
    print(f"\n  ✓ parallel_variants_auto.json: {n_multi} multi-tablet passages found")
else:
    print("  ✗ parallel_variants_auto.json not generated")

# ── Step 5: Passage report ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Diachronic passage report")
print("=" * 60)
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",  str(variants_path),
            "--output", f"{REPO_DIR}/outputs/analysis/passage_reports",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("Skipped — parallel_variants_auto.json not found")

# ── Copy all to Drive ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)

# LM files
for f in sorted(lm_dir.glob("*.json")):
    shutil.copy(f, f"{CHECKPOINTS_DIR}/{f.name}")
    print(f"  ✓ {f.name}")

# Parallel variants
if variants_path.exists():
    shutil.copy(variants_path, f"{CHECKPOINTS_DIR}/parallel_variants_auto.json")
    print(f"  ✓ parallel_variants_auto.json")

# Passage report HTMLs
passage_dir = Path(f"{REPO_DIR}/outputs/analysis/passage_reports")
if passage_dir.exists():
    n = 0
    for f in passage_dir.iterdir():
        if f.is_file() and f.suffix == ".html":
            shutil.copy(f, f"{CHECKPOINTS_DIR}/{f.name}")
            n += 1
    print(f"  ✓ passage_reports/ ({n} HTML files)")

print("\nZone C Cell 1 is ready to run.")

In [ ]:
# ── Zone B Cell 2: Reading order tests ────────────────────────────────────────
# Four entropy tests for rongorongo transcription direction and recto/verso order.
# Tests 3 & 4 need only the corpus — they run immediately.
# Tests 1 & 2 need the sequence model from Zone A Cell 1 — skipped automatically if absent.
#
# Test 1: Conditional entropy asymmetry  H_forward vs H_reverse
# Test 2: N-gram model perplexity asymmetry (trained model, forward vs reversed)
# Test 3: Line-boundary entropy — are line breaks real compositional units?
# Test 4: Recto/verso ordering — resolves Pozdniakov's question from 1958
#
# Outputs written to Drive:
#   reading_order_results.json       ← structured test results (JSON)
#   reading_order_report.html        ← scholar-facing HTML report

import os, subprocess, sys, json, shutil
from pathlib import Path

REPO_DIR        = "/kaggle/working/repo"
CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

RESULTS_JSON = Path(f"{REPO_DIR}/outputs/reading_order_results.json")
REPORT_HTML  = Path(f"{REPO_DIR}/outputs/analysis/reading_order_report.html")
CORPUS_DIR   = Path(f"{DATA_DIR}/corpus")
MODEL_PATH   = Path(f"{REPO_DIR}/outputs/zone_b/sequence_model.json")

if not CORPUS_DIR.exists():
    raise RuntimeError(f"Corpus directory not found: {CORPUS_DIR}\nRun Setup Cell 3 first.")

# ── Tests 3 & 4 (no model required) ──────────────────────────────────────────
print("=" * 60)
print("READING ORDER TESTS 3 & 4 (corpus only)")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "scripts/reading_order_tests.py",
        "--corpus", str(CORPUS_DIR),
        "--tests",  "3", "4",
        "--output", str(RESULTS_JSON),
    ],
    cwd=REPO_DIR,
)

# ── Tests 1 & 2 (sequence model required) ────────────────────────────────────
if MODEL_PATH.exists():
    print("\n" + "=" * 60)
    print("READING ORDER TESTS 1 & 2 (sequence model found)")
    print("=" * 60)
    subprocess.check_call(
        [
            sys.executable, "scripts/reading_order_tests.py",
            "--corpus", str(CORPUS_DIR),
            "--model",  str(MODEL_PATH),
            "--tests",  "1", "2", "3", "4",
            "--output", str(RESULTS_JSON),
        ],
        cwd=REPO_DIR,
    )
else:
    print(f"\n  Sequence model not found at {MODEL_PATH}")
    print("  Tests 1 & 2 skipped. Run Zone A Cell 1 to train the model, then re-run this cell.")

# ── Generate HTML report ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATING READING ORDER REPORT")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "-m", "hackingrongo.results.reading_order_report",
        "--input",  str(RESULTS_JSON),
        "--output", str(REPORT_HTML),
    ],
    cwd=REPO_DIR,
)

# ── Display key results ───────────────────────────────────────────────────────
if RESULTS_JSON.exists():
    results = json.loads(RESULTS_JSON.read_text())
    t4 = results.get("test4", {})
    t3 = results.get("test3", {})
    if t4:
        print("\n" + "=" * 60)
        print("TEST 4  RECTO / VERSO ORDERING")
        print("=" * 60)
        print(f"  {t4.get('verdict_text', '(no verdict)')}")
        ppl_ab2 = t4.get("ppl_ab_bigram");  ppl_ba2 = t4.get("ppl_ba_bigram")
        ppl_ab3 = t4.get("ppl_ab_trigram"); ppl_ba3 = t4.get("ppl_ba_trigram")
        if all(v is not None for v in [ppl_ab2, ppl_ba2, ppl_ab3, ppl_ba3]):
            print(f"  Bigram  LOO-PPL   a→b : {ppl_ab2:.2f}   b→a : {ppl_ba2:.2f}")
            print(f"  Trigram LOO-PPL   a→b : {ppl_ab3:.2f}   b→a : {ppl_ba3:.2f}")
    if t3:
        print("\nTEST 3  LINE BOUNDARY ENTROPY")
        print(f"  {t3.get('verdict_text', '(no verdict)')}")

# ── Copy to Drive ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)
for src, name in [
    (RESULTS_JSON, "reading_order_results.json"),
    (REPORT_HTML,  "reading_order_report.html"),
]:
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{name}")
        print(f"  ✓ {name}  ({src.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  — {name}  (not generated)")

print("\nZone C Cell 1 is ready to run.")

In [ ]:
# ── Zone C Cell 1: MCMC and beam search decipherment ─────────────────────────
# Runs the probabilistic key search over the sign→phoneme space.
# Requires Zone B Cell 1 to have completed (language models must exist).
#
# SMOKE_TEST = True  → fast validation (2 chains × 500 iterations, Tablet D only)
#                      Takes ~3 min. Run this first to confirm Zone C is working.
# MEDIUM_RUN = True  → 2 chains × 5k iterations, full corpus (~30 min on CPU)
# Both False         → full run (4 chains × 50k iterations, ~2 hrs on GPU)
#
# Gates: acceptance rates 0.05–0.95, finite log-posterior, LMs loaded.
#
# Outputs written to Drive:
#   decipherment/ranking.json            ← ranked hypotheses
#   decipherment/ranking.csv/.md         ← tabular formats
#   decipherment/decipherment_report.html ← scholar-facing HTML report

import os, subprocess, sys, json, shutil, math
from pathlib import Path

REPO_DIR         = "/kaggle/working/repo"
CHECKPOINTS_DIR  = "/kaggle/working/checkpoints"
DECIPHERMENT_OUT = f"{REPO_DIR}/outputs/decipherment"

SMOKE_TEST  = True   # ← 2 chains × 500 iter, Tablet D only (~3 min)
MEDIUM_RUN  = False  # ← 2 chains × 5000 iter, full corpus (~30 min on CPU)
# Both False → full run: 4 chains × 50k iter (~2 hrs on GPU)

os.chdir(REPO_DIR)
Path(DECIPHERMENT_OUT).mkdir(parents=True, exist_ok=True)

# ── Gate check: LMs present ───────────────────────────────────────────────────
lm_dir = Path(f"{REPO_DIR}/data/language_models")
missing_lms = [
    name for name in ["pre_contact_lm.json", "post_contact_lm.json"]
    if not (lm_dir / name).exists()
]
if missing_lms:
    raise RuntimeError(
        f"Missing language models: {missing_lms}\n"
        "Run Zone B Cell 1 first to build the language models."
    )

cmd = [sys.executable, "scripts/run_decipherment.py", "hydra.job.chdir=false"]
if SMOKE_TEST:
    cmd.append("--smoke-test")
elif MEDIUM_RUN:
    cmd += [
        "zone_c.mcmc.num_chains=2",
        "zone_c.mcmc.num_iterations=5000",
        "zone_c.mcmc.burn_in=500",
        "zone_c.mcmc.thin=10",
    ]
# else: full run uses config defaults

mode_str = (
    "SMOKE TEST (2 chains × 500 iter, Tablet D only)" if SMOKE_TEST else
    "MEDIUM RUN (2 chains × 5k iter, full corpus)"    if MEDIUM_RUN else
    "FULL RUN (4 chains × 50k iter)"
)

print("=" * 60)
print(f"ZONE C DECIPHERMENT — {mode_str}")
print("=" * 60)

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Zone C failed (exit {proc.returncode})")

# ── Parse and display results ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)

ranking_path = Path(DECIPHERMENT_OUT) / "ranking.json"
if ranking_path.exists():
    ranking = json.loads(ranking_path.read_text())
    hyps = ranking.get("hypotheses", [])
    print(f"  Hypotheses ranked: {len(hyps)}")

    if hyps:
        top = hyps[0]
        print(f"\nTop hypothesis: {top.get('hypothesis_id', '?')}")
        print(f"  Overall LM score:    {top.get('overall_lm_score', 0):.4f}")
        print(f"  MCMC log-posterior:  {top.get('mcmc_log_posterior', 0):.4f}")
        print(f"  Beam score:          {top.get('beam_score', 0):.4f}")

        # Gate checks
        print("\n── Gate checks ─────────────────────────────────────────")
        lm_score = top.get("overall_lm_score", -math.inf)
        oov_floor = -15.0 * 3  # approximate OOV floor for trigrams
        if math.isfinite(lm_score) and lm_score > oov_floor:
            print(f"  ✓ LM score {lm_score:.4f} is above OOV floor ({oov_floor:.1f})")
        else:
            print(f"  ✗ LM score {lm_score:.4f} — at or below OOV floor")
            print("    Zone C may be scoring against empty LMs — rebuild with Zone B Cell 1")

        print(f"\nTop 5 sign→phoneme assignments:")
        assignments = top.get("phoneme_assignments", [])[:5]
        for a in assignments:
            print(f"  {a.get('sign_code','?'):8s} → {a.get('phoneme','?'):6s}  "
                  f"conf={a.get('confidence',0):.3f}  n={a.get('evidence_count',0)}")

    if SMOKE_TEST:
        print("\nSmoke test passed. Set SMOKE_TEST = False and rerun for full results.")
else:
    print("  ranking.json not found — check logs above")

# ── Copy outputs to Drive ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)
for fname in ["ranking.json", "ranking.csv", "ranking.md", "decipherment_report.html"]:
    src = Path(DECIPHERMENT_OUT) / fname
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{fname}")
        print(f"  ✓ {fname}  {src.stat().st_size/1024:.0f} KB")
    else:
        print(f"  — {fname}  (not generated)")

print("\nRun Final Cell 1 to download all results.")

In [ ]:
# ── Zone C Cell 2: Gumbel-Softmax gradient refinement of MCMC assignments ─────
# Refines the top-K MCMC hypotheses with ~100 gradient steps through a
# differentiable soft-LM scorer (add-α smoothed bigram transition matrix).
# Adds refined_assignments, refined_soft_score, and refined_lm_score to
# each hypothesis in ranking.json for honest comparison with the MCMC baseline.
#
# Requires Zone C Cell 1 to have completed (ranking.json must exist).
#
# N_STEPS = 10 → fast smoke-test validation (~5 sec)
# N_STEPS = 100 → full refinement (~30 sec per hypothesis on CPU)
# TOP_K = 5 → refine all 5 ranked hypotheses

import os, subprocess, sys, json, math
from pathlib import Path

REPO_DIR         = "/kaggle/working/repo"
CHECKPOINTS_DIR  = "/kaggle/working/checkpoints"
DECIPHERMENT_OUT = f"{REPO_DIR}/outputs/decipherment"

N_STEPS = 100   # gradient steps per hypothesis
TOP_K   = 5     # number of top hypotheses to refine

os.chdir(REPO_DIR)

# ── Gate check: ranking.json present ─────────────────────────────────────────
ranking_path = Path(DECIPHERMENT_OUT) / "ranking.json"
if not ranking_path.exists():
    raise RuntimeError(
        "ranking.json not found — run Zone C Cell 1 first."
    )

lm_dir = Path(f"{REPO_DIR}/data/language_models")

cmd = [
    sys.executable, "scripts/refine_assignments.py",
    "--ranking",      str(ranking_path),
    "--lm-dir",       str(lm_dir),
    "--n-steps",      str(N_STEPS),
    "--top-k",        str(TOP_K),
    "--project-root", REPO_DIR,
    "--output",       str(ranking_path),   # overwrite in-place
]

print("=" * 60)
print(f"ZONE C GRADIENT REFINEMENT — {N_STEPS} steps × top-{TOP_K} hypotheses")
print("=" * 60)

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Gradient refinement failed (exit {proc.returncode})")

# ── Parse and display refinement results ─────────────────────────────────────
print("\n" + "=" * 60)
print("REFINEMENT RESULTS")
print("=" * 60)

ranking = json.loads(ranking_path.read_text())
hyps = ranking.get("hypotheses", [])[:TOP_K]

print(f"{'Rank':<6}{'Hypothesis':<12}{'Orig LM':>12}{'Refined LM':>12}{'Delta':>10}")
print("-" * 52)
for i, h in enumerate(hyps):
    orig  = h.get("overall_lm_score")
    ref   = h.get("refined_lm_score")
    delta = h.get("refinement_delta_lm")
    hid   = h.get("hypothesis_id", f"H{i:04d}")
    orig_s  = f"{orig:.4f}"  if orig  is not None else "N/A"
    ref_s   = f"{ref:.4f}"   if ref   is not None else "N/A"
    delta_s = f"{delta:+.4f}" if delta is not None else "N/A"
    print(f"{i+1:<6}{hid:<12}{orig_s:>12}{ref_s:>12}{delta_s:>10}")

print("\nRefinement complete. ranking.json updated with refined_assignments.")
import shutil
shutil.copy(ranking_path, f"{CHECKPOINTS_DIR}/ranking.json")
print(f"  ✓ ranking.json copied to checkpoints ({ranking_path.stat().st_size/1024:.0f} KB)")


In [ ]:
# ── Final Cell 1: Copy outputs to /kaggle/working for download ────────────────
# Run as the final cell. Everything in /kaggle/working/ appears in the
# Output tab (right sidebar) and can be downloaded or committed.
# Hit "Save Version" (top right) to make outputs permanent.

import shutil
from pathlib import Path

REPO_DIR        = "/kaggle/working/repo"
CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
OUTPUT_DIR      = Path("/kaggle/working")
HTML_DIR        = OUTPUT_DIR / "html_reports"

print("── Copying outputs to /kaggle/working ───────────────────────")

# Helper: flat-copy files to /kaggle/working root.
def _copy_flat(src: Path, dest_name: str | None = None, note_if_missing: str | None = None):
    if src.exists():
        dest = OUTPUT_DIR / (dest_name or src.name)
        shutil.copy(src, dest)
        print(f"  ✓ {dest.name:45s} {src.stat().st_size/1024:.0f} KB")
        return True
    if note_if_missing:
        print(f"  — {(dest_name or src.name):45s} {note_if_missing}")
    return False

# ── Checkpoints and embeddings ────────────────────────────────────────────────
for f in sorted(Path(CHECKPOINTS_DIR).glob("*.pt")):
    shutil.copy(f, OUTPUT_DIR / f.name)
    print(f"  ✓ {f.name:45s} {f.stat().st_size/1024/1024:.1f} MB")

# ── Zone A: UMAP + cluster data ───────────────────────────────────────────────
for fname in [
    "umap_embeddings.png",
    "cluster_vs_barthel.csv",
    "cluster_vs_barthel.json",
]:
    src = Path(f"{REPO_DIR}/outputs/analysis/{fname}")
    _copy_flat(src)

# ── Zone B: Compound detection ────────────────────────────────────────────────
for fname in [
    "compound_candidates.json",
    "compound_validation.json",
    "compound_report.html",
]:
    src = Path(f"{REPO_DIR}/outputs/analysis/{fname}")
    _copy_flat(src)

# ── Zone B: Parallel passage data ─────────────────────────────────────────────
_copy_flat(
    Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json"),
    note_if_missing="not generated (run Zone B cell)",
)

# ── Zone B: Astronomical analysis ────────────────────────────────────────────
_copy_flat(
    Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json"),
    note_if_missing="not generated (run Zone A Cell 2 Step 8)",
)

# ── Zone C: Decipherment outputs ──────────────────────────────────────────────
decipher_dir = Path(f"{REPO_DIR}/outputs/decipherment")
for fname in ["ranking.json", "ranking.csv", "ranking.md", "decipherment_report.html"]:
    _copy_flat(
        decipher_dir / fname,
        note_if_missing="not generated (run Zone C cell)",
    )

# ── Zone C: Per-hypothesis JSON files ─────────────────────────────────────────
if decipher_dir.exists():
    hyp_files = sorted(decipher_dir.glob("hypothesis_*.json"))
    if hyp_files:
        for f in hyp_files[:10]:  # cap at 10 to avoid flooding output tab
            shutil.copy(f, OUTPUT_DIR / f.name)
        print(f"  ✓ {'hypothesis_*.json (top 10)':45s} {len(hyp_files)} total")

# ── Language models (needed to resume Zone C next session) ────────────────────
lm_dir = Path(f"{DATA_DIR}/language_models")
if lm_dir.exists():
    lm_files = sorted(lm_dir.glob("*.json"))
    lm_total_kb = 0
    for f in lm_files:
        shutil.copy(f, OUTPUT_DIR / f.name)
        lm_total_kb += f.stat().st_size / 1024
    print(f"  ✓ {'language_models/ (all .json)':45s} {len(lm_files)} files, {lm_total_kb:.0f} KB total")
else:
    print(f"  — {'language_models/':45s} not built (run Zone B cell)")

# ── Zone B: Sensitivity + contact analysis ────────────────────────────────────
for fname in [
    "sensitivity_analysis.json",
    "contact_partition.json",
    "contact_partition_bipartite.html",
    "sequence_model.json",
]:
    src = Path(f"{REPO_DIR}/outputs/{fname}")
    _copy_flat(src)

# ── Zone B: Sequence model + reconstruction ───────────────────────────────────
for fname in ["sequence_model.json", "tablet_f_reconstruction.json"]:
    src = Path(f"{REPO_DIR}/outputs/zone_b/{fname}")
    _copy_flat(src)

# ── Layer 5Q: Quantum hardness analysis ───────────────────────────────────────
_copy_flat(
    Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json"),
    note_if_missing="not generated (run Zone A Cell 2 Step 8b)",
)

# ── NEW: Sync ALL HTML reports recursively to /kaggle/working/html_reports ────
print("\n── Syncing all HTML reports (recursive) ─────────────────────")
if HTML_DIR.exists():
    shutil.rmtree(HTML_DIR)
HTML_DIR.mkdir(parents=True, exist_ok=True)

html_sources = sorted(
    [
        p
        for p in Path(f"{REPO_DIR}/outputs").rglob("*")
        if p.is_file() and p.suffix.lower() == ".html"
    ]
)

for src in html_sources:
    rel = src.relative_to(Path(f"{REPO_DIR}/outputs"))
    dest = HTML_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

# Also include any HTML reports that may already exist in checkpoints.
for src in sorted(Path(CHECKPOINTS_DIR).rglob("*.html")):
    rel = src.relative_to(Path(CHECKPOINTS_DIR))
    dest = HTML_DIR / Path("checkpoints") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

# Create a single ZIP for one-click download from Kaggle Output panel.
zip_base = OUTPUT_DIR / "html_reports_all"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=HTML_DIR))

print(f"  ✓ Synced HTML reports into: {HTML_DIR}")
print(f"  ✓ HTML bundle ready: {zip_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
all_files = [f for f in OUTPUT_DIR.iterdir() if f.is_file()]
total_mb = sum(f.stat().st_size for f in all_files) / 1024 / 1024
html_files = sorted([f for f in HTML_DIR.rglob("*.html") if f.is_file()])

print(f"\n── Summary ──────────────────────────────────────────────────")
print(f"  Root output files: {len(all_files)}")
print(f"  Root output size:  {total_mb:.1f} MB")
print(f"  HTML reports:      {len(html_files)}")
for f in html_files:
    print(f"    {f.relative_to(HTML_DIR)}")

print("\nDownload instructions:")
print("  1) Open Output tab (right sidebar)")
print("  2) Download html_reports_all.zip for all HTML reports at once")
print("  3) Or open html_reports/ to download individual HTML files")
print("  4) Save Version (top right) to commit outputs permanently")